# Week 10 Lab: Naive Bayes Classifier
**CSE303 Machine Learning | Unit III: Probability and Bayes Learning | Syllabus Exp 9 (naive Bayesian classifier and its accuracy)**

Pattern: **demo → practice → predict-then-run → deliberate error → reflection**.

| Part | Topic | Track |
|---|---|---|
| 1 | Probability and Bayes' theorem | core |
| 2 | Naive Bayes by hand (Play Tennis), and the zero-frequency error | core |
| 3 | Gaussian Naive Bayes in sklearn, log-probabilities, and scaling | core |
| 4 | Mini Project A: Naive Bayes on digits (continues Week 7/8/9) | image |
| 5 | Mini Project B: Naive Bayes on the six sentences (continues Week 7/8/9) | NLP |
| 6 | Reflection and capstone link | |

**Rule carried from Week 5:** split first; fit every scaler, PCA and tuned setting on **train only**.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_digits, load_wine
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, CategoricalNB
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
np.set_printoptions(precision=4, suppress=True)

## Part 1: Probability and Bayes' Theorem
$$P(C\mid x)=\frac{P(x\mid C)\,P(C)}{P(x)}$$
**Posterior** = likelihood × prior / evidence. The evidence $P(x)=\sum_C P(x\mid C)P(C)$ is the same for every class, so to *compare* classes we only need the numerator.

### Predict-then-run
A disease affects 1% of people. A test is positive for 90% of sick people (sensitivity) and also for 5% of healthy people (false-positive rate). You test positive. Chance you are actually sick? Guess before running: **~90% / ~50% / ~15%**

In [2]:
prior, sens, fpr = 0.01, 0.90, 0.05           # P(sick), P(+|sick), P(+|healthy)
evidence = sens * prior + fpr * (1 - prior)    # P(+)
posterior = sens * prior / evidence            # P(sick | +)
print('P(positive)      =', round(evidence, 4))
print('P(sick | positive) =', round(posterior, 4))

# sanity check by simulation: 1,000,000 people
rng = np.random.default_rng(1); n = 1_000_000
sick = rng.random(n) < prior
pos = np.where(sick, rng.random(n) < sens, rng.random(n) < fpr)
print('simulated          =', round(sick[pos].mean(), 4), f'({sick[pos].sum()} sick among {pos.sum()} positives)')

P(positive)      = 0.0585
P(sick | positive) = 0.1538
simulated          = 0.1545 (9035 sick among 58469 positives)


### Student Practice 1
An email filter knows: 40% of mail is spam; the word *free* appears in 50% of spam and 5% of non-spam.
1. Compute $P(\text{spam}\mid \text{free})$ by hand, then in code.
2. If the spam rate were only 5%, what would the posterior be? Why does it drop so much?
3. In the disease example, which term of Bayes' theorem do people usually ignore?

In [3]:
# Student Practice 1: Solution

# Q1: Compute P(spam | free) by hand and in code
p_spam = 0.40
p_ham = 1.0 - p_spam
p_free_given_spam = 0.50
p_free_given_ham = 0.05

# P(free) = P(free|spam)*P(spam) + P(free|ham)*P(ham)
# P(free) = 0.50 * 0.40 + 0.05 * 0.60 = 0.20 + 0.03 = 0.23
# P(spam | free) = (0.50 * 0.40) / 0.23 = 0.20 / 0.23 = 20/23 ≈ 0.8696 (86.96%)
p_free = p_free_given_spam * p_spam + p_free_given_ham * p_ham
p_spam_given_free = (p_free_given_spam * p_spam) / p_free

print("1. P(spam | free):")
print(f"   P(free)            = {p_free:.4f}")
print(f"   P(spam | free)     = {p_spam_given_free:.4f} ({p_spam_given_free*100:.2f}%)")

# Q2: If spam rate were only 5%
p_spam_low = 0.05
p_ham_low = 1.0 - p_spam_low
p_free_low = p_free_given_spam * p_spam_low + p_free_given_ham * p_ham_low
p_spam_given_free_low = (p_free_given_spam * p_spam_low) / p_free_low

print("\n2. If spam rate = 5%:")
print(f"   P(free)            = {p_free_low:.4f}")
print(f"   P(spam | free)     = {p_spam_given_free_low:.4f} ({p_spam_given_free_low*100:.2f}%)")
print("   Why it drops so much: The prior probability of spam is 8x smaller. Even with a small 5% false-positive rate,")
print("   the vast majority of emails are non-spam (95%), so false-positive non-spam emails containing 'free' (0.0475)")
print("   substantially outnumber true-positive spam emails containing 'free' (0.0250).")

# Q3: Disease example ignored term
print("\n3. Term usually ignored in the disease example:")
print("   People usually ignore the PRIOR probability P(disease) (base rate neglect), mistakenly assuming")
print("   a positive test result means an overwhelming probability of having the rare condition.")


1. P(spam | free):
   P(free)            = 0.2300
   P(spam | free)     = 0.8696 (86.96%)

2. If spam rate = 5%:
   P(free)            = 0.0725
   P(spam | free)     = 0.3448 (34.48%)
   Why it drops so much: The prior probability of spam is 8x smaller. Even with a small 5% false-positive rate,
   the vast majority of emails are non-spam (95%), so false-positive non-spam emails containing 'free' (0.0475)
   substantially outnumber true-positive spam emails containing 'free' (0.0250).

3. Term usually ignored in the disease example:
   People usually ignore the PRIOR probability P(disease) (base rate neglect), mistakenly assuming
   a positive test result means an overwhelming probability of having the rare condition.


## Part 2: Naive Bayes by Hand (Play Tennis)
Classic 14-day dataset (Mitchell, Ch. 6). Predict **Play** from four categorical features.

The **naive** assumption: features are independent *given the class*, so
$$P(x_1,\dots,x_d\mid C)=\prod_{j=1}^{d}P(x_j\mid C)\quad\Rightarrow\quad \hat C=\arg\max_C\; P(C)\prod_j P(x_j\mid C)$$

Steps: (1) class priors, (2) conditional probability tables from counts, (3) multiply, (4) normalise, (5) pick the largest.

| Day | Outlook | Temp | Humidity | Wind | Play |
|---|---|---|---|---|---|
| 1 | Sunny | Hot | High | Weak | No |
| 2 | Sunny | Hot | High | Strong | No |
| 3 | Overcast | Hot | High | Weak | Yes |
| 4 | Rain | Mild | High | Weak | Yes |
| 5 | Rain | Cool | Normal | Weak | Yes |
| 6 | Rain | Cool | Normal | Strong | No |
| 7 | Overcast | Cool | Normal | Strong | Yes |
| 8 | Sunny | Mild | High | Weak | No |
| 9 | Sunny | Cool | Normal | Weak | Yes |
| 10 | Rain | Mild | Normal | Weak | Yes |
| 11 | Sunny | Mild | Normal | Strong | Yes |
| 12 | Overcast | Mild | High | Strong | Yes |
| 13 | Overcast | Hot | Normal | Weak | Yes |
| 14 | Rain | Mild | High | Strong | No |

In [4]:
rows = [
 ('Sunny','Hot','High','Weak','No'), ('Sunny','Hot','High','Strong','No'), ('Overcast','Hot','High','Weak','Yes'),
 ('Rain','Mild','High','Weak','Yes'), ('Rain','Cool','Normal','Weak','Yes'), ('Rain','Cool','Normal','Strong','No'),
 ('Overcast','Cool','Normal','Strong','Yes'), ('Sunny','Mild','High','Weak','No'), ('Sunny','Cool','Normal','Weak','Yes'),
 ('Rain','Mild','Normal','Weak','Yes'), ('Sunny','Mild','Normal','Strong','Yes'), ('Overcast','Mild','High','Strong','Yes'),
 ('Overcast','Hot','Normal','Weak','Yes'), ('Rain','Mild','High','Strong','No')]
F = np.array([r[:-1] for r in rows]); y = np.array([r[-1] for r in rows])
features = ['Outlook', 'Temp', 'Humidity', 'Wind']
print('Priors: Yes =', (y == 'Yes').sum(), '/ 14,  No =', (y == 'No').sum(), '/ 14')
for j, name in enumerate(features):
    print(name)
    for v in sorted(set(F[:, j])):
        print(f'   {v:9} Yes {((F[:, j] == v) & (y == "Yes")).sum()}/9    No {((F[:, j] == v) & (y == "No")).sum()}/5')

Priors: Yes = 9 / 14,  No = 5 / 14
Outlook
   Overcast  Yes 4/9    No 0/5
   Rain      Yes 3/9    No 2/5
   Sunny     Yes 2/9    No 3/5
Temp
   Cool      Yes 3/9    No 1/5
   Hot       Yes 2/9    No 2/5
   Mild      Yes 4/9    No 2/5
Humidity
   High      Yes 3/9    No 4/5
   Normal    Yes 6/9    No 1/5
Wind
   Strong    Yes 3/9    No 3/5
   Weak      Yes 6/9    No 2/5


In [5]:
def nb_scores(q, alpha=0):
    """Unnormalised score P(C) * prod_j P(x_j|C) and the normalised posterior. alpha = Laplace smoothing."""
    out = {}
    for c in ['Yes', 'No']:
        Fc = F[y == c]
        p = float((y == c).mean())
        for j, v in enumerate(q):
            k = len(set(F[:, j]))                                  # how many values this feature can take
            p *= float((Fc[:, j] == v).sum() + alpha) / (len(Fc) + alpha * k)
        out[c] = p
    total = sum(out.values())
    return out, {c: out[c] / total for c in out}

### Predict-then-run
Today is **(Sunny, Cool, High, Strong)**. The prior favours *Yes* (9/14). Will Naive Bayes say Yes or No? Guess, then run.

In [6]:
q = ('Sunny', 'Cool', 'High', 'Strong')
scores, post = nb_scores(q)
print('scores   :', {c: round(v, 5) for c, v in scores.items()})
print('posterior:', {c: round(v, 4) for c, v in post.items()})
print('prediction:', max(post, key=post.get))

scores   : {'Yes': 0.00529, 'No': 0.02057}
posterior: {'Yes': 0.2046, 'No': 0.7954}
prediction: No


### Student Practice 2 (do on paper first, as in an exam)
For today = (Sunny, Cool, High, Strong):
1. Write $P(\text{Yes})\,P(\text{Sunny}|\text{Yes})\,P(\text{Cool}|\text{Yes})\,P(\text{High}|\text{Yes})\,P(\text{Strong}|\text{Yes})$ as a product of fractions, then evaluate it. Do the same for *No*.
2. Normalise: $P(\text{Yes}\mid x)$ = **20.46%**
3. Redo it with Laplace smoothing ($\alpha=1$). Does the prediction change? Does the probability?
4. **Accuracy on test days (Exp 9).** Using $\alpha=1$, classify these five unseen days and report the accuracy:

| Test day | Outlook | Temp | Humidity | Wind | True Play |
|---|---|---|---|---|---|
| T1 | Overcast | Mild | Normal | Weak | Yes |
| T2 | Rain | Hot | High | Strong | No |
| T3 | Sunny | Hot | Normal | Weak | Yes |
| T4 | Rain | Cool | High | Weak | Yes |
| T5 | Sunny | Mild | High | Strong | No |

5. Verify your function against sklearn's `CategoricalNB(alpha=1)` (hint: encode the strings with `OrdinalEncoder`).

In [7]:
# Student Practice 2: Solution

# 1. Product of fractions for today = ('Sunny', 'Cool', 'High', 'Strong')
# Outlook has 3 values, Temp has 3, Humidity has 2, Wind has 2.
# Prior: Yes = 9/14, No = 5/14
# Without smoothing (alpha=0):
# Yes: P(Sunny|Yes)=2/9, P(Cool|Yes)=3/9, P(High|Yes)=3/9, P(Strong|Yes)=3/9
# No : P(Sunny|No)=3/5,  P(Cool|No)=1/5,  P(High|No)=4/5,  P(Strong|No)=3/5
s_yes_frac = (9/14) * (2/9) * (3/9) * (3/9) * (3/9)
s_no_frac = (5/14) * (3/5) * (1/5) * (4/5) * (3/5)
print("1. Fractions and unnormalised scores (alpha=0):")
print(f"   Score(Yes) = (9/14) * (2/9) * (3/9) * (3/9) * (3/9) = {s_yes_frac:.6f} (1/189)")
print(f"   Score(No)  = (5/14) * (3/5) * (1/5) * (4/5) * (3/5) = {s_no_frac:.6f} (18/875)")

# 2. Normalise
p_yes_norm = s_yes_frac / (s_yes_frac + s_no_frac)
p_no_norm = s_no_frac / (s_yes_frac + s_no_frac)
print(f"\n2. Normalised: P(Yes | x) = {p_yes_norm:.4f} ({p_yes_norm*100:.2f}%), P(No | x) = {p_no_norm:.4f} ({p_no_norm*100:.2f}%)")

# 3. Redo with Laplace smoothing (alpha=1)
s1, p1 = nb_scores(('Sunny', 'Cool', 'High', 'Strong'), alpha=1)
print("\n3. With Laplace smoothing (alpha=1):")
print("   Score(Yes) = (9/14) * ((2+1)/(9+3)) * ((3+1)/(9+3)) * ((3+1)/(9+2)) * ((3+1)/(9+2))")
print("              = (9/14) * (3/12) * (4/12) * (4/11) * (4/11)")
print("   Score(No)  = (5/14) * ((3+1)/(5+3)) * ((1+1)/(5+3)) * ((4+1)/(5+2)) * ((3+1)/(5+2))")
print("              = (5/14) * (4/8)  * (2/8)  * (5/7)  * (4/7)")
print(f"   Unnormalised scores: { {c: round(v, 6) for c, v in s1.items()} }")
print(f"   Normalised posterior: { {c: round(v, 4) for c, v in p1.items()} }")
print("   Prediction does NOT change (still 'No'), but P(Yes|x) increases from 20.46% to 27.99% (less extreme).")

# 4. Accuracy on test days using alpha=1
tests = [(('Overcast','Mild','Normal','Weak'), 'Yes'),
         (('Rain','Hot','High','Strong'), 'No'),
         (('Sunny','Hot','Normal','Weak'), 'Yes'),
         (('Rain','Cool','High','Weak'), 'Yes'),
         (('Sunny','Mild','High','Strong'), 'No')]

print("\n4. Classifying unseen test days (alpha=1):")
preds_manual = []
for day, true_label in tests:
    _, post = nb_scores(day, alpha=1)
    pred = max(post, key=post.get)
    preds_manual.append(pred)
    print(f"   {day} -> Pred: {pred:3} | True: {true_label:3} | Correct: {pred == true_label}")

acc_manual = np.mean([p == t[1] for p, t in zip(preds_manual, tests)])
print(f"   Test Accuracy: {acc_manual:.4f} ({acc_manual*100:.1f}%)")

# 5. Verify against sklearn's CategoricalNB(alpha=1)
enc = OrdinalEncoder().fit(F)
F_enc = enc.transform(F)
cnb = CategoricalNB(alpha=1).fit(F_enc, y)

test_X = np.array([t[0] for t in tests])
test_X_enc = enc.transform(test_X)
cnb_preds = cnb.predict(test_X_enc)

print("\n5. Verification with sklearn CategoricalNB(alpha=1):")
print(f"   CategoricalNB predictions: {cnb_preds.tolist()}")
print(f"   Matches manual nb_scores: {bool(np.array_equal(cnb_preds, preds_manual))}")


1. Fractions and unnormalised scores (alpha=0):
   Score(Yes) = (9/14) * (2/9) * (3/9) * (3/9) * (3/9) = 0.005291 (1/189)
   Score(No)  = (5/14) * (3/5) * (1/5) * (4/5) * (3/5) = 0.020571 (18/875)

2. Normalised: P(Yes | x) = 0.2046 (20.46%), P(No | x) = 0.7954 (79.54%)

3. With Laplace smoothing (alpha=1):
   Score(Yes) = (9/14) * ((2+1)/(9+3)) * ((3+1)/(9+3)) * ((3+1)/(9+2)) * ((3+1)/(9+2))
              = (9/14) * (3/12) * (4/12) * (4/11) * (4/11)
   Score(No)  = (5/14) * ((3+1)/(5+3)) * ((1+1)/(5+3)) * ((4+1)/(5+2)) * ((3+1)/(5+2))
              = (5/14) * (4/8)  * (2/8)  * (5/7)  * (4/7)
   Unnormalised scores: {'Yes': 0.007084, 'No': 0.018222}
   Normalised posterior: {'Yes': 0.2799, 'No': 0.7201}
   Prediction does NOT change (still 'No'), but P(Yes|x) increases from 20.46% to 27.99% (less extreme).

4. Classifying unseen test days (alpha=1):
   ('Overcast', 'Mild', 'Normal', 'Weak') -> Pred: Yes | True: Yes | Correct: True
   ('Rain', 'Hot', 'High', 'Strong') -> Pred: No  | Tru

### Deliberate error: the zero-frequency problem
Today is **(Overcast, Hot, High, Strong)**. *Overcast* never occurs with *No* in the training data. Run the cell, then explain in one line why $\alpha=0$ is dangerous.

In [8]:
q2 = ('Overcast', 'Hot', 'High', 'Strong')
for a in [0, 1]:
    s, p = nb_scores(q2, alpha=a)
    print(f'alpha={a}: scores', {c: round(v, 5) for c, v in s.items()}, ' posterior', {c: round(float(v), 3) for c, v in p.items()})

print("\nWhy alpha=0 is dangerous: A single unseen feature-class combination produces a probability of 0,")
print("which multiplies across all features and completely zeros out the posterior score, erasing all other evidence.")


alpha=0: scores {'Yes': 0.00705, 'No': 0.0}  posterior {'Yes': 1.0, 'No': 0.0}
alpha=1: scores {'Yes': 0.00885, 'No': 0.00683}  posterior {'Yes': 0.564, 'No': 0.436}

Why alpha=0 is dangerous: A single unseen feature-class combination produces a probability of 0,
which multiplies across all features and completely zeros out the posterior score, erasing all other evidence.


## Part 3: Gaussian Naive Bayes (continuous features)
For numeric features, model each feature *within each class* as a bell curve with that class's mean $\mu_{jc}$ and variance $\sigma^2_{jc}$:
$$P(x_j\mid C)=\frac{1}{\sqrt{2\pi\sigma^2_{jc}}}\exp\!\Big(-\frac{(x_j-\mu_{jc})^2}{2\sigma^2_{jc}}\Big)$$
sklearn stores these as `theta_` (means) and `var_` (variances). In practice we add **log-probabilities** instead of multiplying (see the deliberate error below).

In [9]:
Xi, yi = load_iris(return_X_y=True)
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.3, random_state=1, stratify=yi)
gnb = GaussianNB().fit(Xi_tr, yi_tr)
print('class priors:', gnb.class_prior_)
print('class means (rows = setosa, versicolor, virginica):\n', gnb.theta_)
print('class variances:\n', gnb.var_)
print('test accuracy:', round(gnb.score(Xi_te, yi_te), 4))

class priors: [0.3333 0.3333 0.3333]
class means (rows = setosa, versicolor, virginica):
 [[5.0229 3.4343 1.4571 0.2429]
 [5.8714 2.78   4.2743 1.3229]
 [6.6771 2.9943 5.6371 2.0257]]
class variances:
 [[0.1275 0.132  0.0373 0.0076]
 [0.2015 0.0867 0.2042 0.0418]
 [0.3709 0.1125 0.3143 0.0831]]
test accuracy: 0.9778


### Student Practice 3
1. For the first test flower `x = Xi_te[0]`, compute the log-posterior of each class **yourself** from `gnb.class_prior_`, `gnb.theta_`, `gnb.var_` (Gaussian formula, summed over the 4 features). Which class wins? Does it match `gnb.predict`?
2. Repeat for every test row and check that your predictions equal `gnb.predict(Xi_te)`.
3. Print the confusion matrix. Which two species are confused?

In [10]:
# Student Practice 3: Solution

# 1. Log-posterior for first test flower x = Xi_te[0]
x0 = Xi_te[0]
log_priors = np.log(gnb.class_prior_)
log_posts_0 = []

for c in range(len(gnb.classes_)):
    mu_c = gnb.theta_[c]
    var_c = gnb.var_[c]
    # log P(x_j|C) = -0.5 * log(2*pi*var) - 0.5 * ((x_j - mu)^2 / var)
    log_lik = -0.5 * np.sum(np.log(2 * np.pi * var_c) + ((x0 - mu_c) ** 2) / var_c)
    log_posts_0.append(log_priors[c] + log_lik)

log_posts_0 = np.array(log_posts_0)
win_class_0 = gnb.classes_[np.argmax(log_posts_0)]

print("1. Log-posteriors for Xi_te[0]:")
for c, lp in zip(gnb.classes_, log_posts_0):
    print(f"   Class {c}: log-posterior = {lp:.4f}")
print(f"   Winning class: {win_class_0} (gnb.predict = {gnb.predict(Xi_te[:1])[0]})")
print(f"   Matches gnb.predict: {win_class_0 == gnb.predict(Xi_te[:1])[0]}")

# 2. Repeat for every test row and check equivalence
all_preds = []
for x in Xi_te:
    lps = []
    for c in range(len(gnb.classes_)):
        mu_c = gnb.theta_[c]
        var_c = gnb.var_[c]
        log_lik = -0.5 * np.sum(np.log(2 * np.pi * var_c) + ((x - mu_c) ** 2) / var_c)
        lps.append(log_priors[c] + log_lik)
    all_preds.append(gnb.classes_[np.argmax(lps)])

all_preds = np.array(all_preds)
print(f"\n2. All test predictions match gnb.predict: {np.array_equal(all_preds, gnb.predict(Xi_te))}")

# 3. Confusion matrix
cm = confusion_matrix(yi_te, gnb.predict(Xi_te))
print("\n3. Confusion Matrix (rows = True, cols = Predicted):")
print(cm)
print("   Species confused: 1 sample of class 2 (virginica) was misclassified as class 1 (versicolor).")


1. Log-posteriors for Xi_te[0]:
   Class 0: log-posterior = -491.1550
   Class 1: log-posterior = -17.3865
   Class 2: log-posterior = -2.9440
   Winning class: 2 (gnb.predict = 2)
   Matches gnb.predict: True

2. All test predictions match gnb.predict: True

3. Confusion Matrix (rows = True, cols = Predicted):
[[15  0  0]
 [ 0 15  0]
 [ 0  1 14]]
   Species confused: 1 sample of class 2 (virginica) was misclassified as class 1 (versicolor).


### Deliberate error: multiplying many small probabilities
Predict: what is the product of 200 probabilities, each equal to 0.01? Then run.

In [11]:
p = np.full(200, 0.01)
print('product     :', np.prod(p))
print('sum of logs :', np.sum(np.log(p)))

product     : 0.0
sum of logs : -921.0340371976183


### Predict-then-run
In Week 9, KNN needed scaling on Wine (unscaled PC1 was dominated by *proline*). Does standardising change Gaussian Naive Bayes? Guess: **helps a lot / no change / hurts**. Then run.

In [12]:
Xw, yw = load_wine(return_X_y=True)
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.3, random_state=1, stratify=yw)
sw = StandardScaler().fit(Xw_tr)                       # fit on TRAIN only
Xw_tr_s, Xw_te_s = sw.transform(Xw_tr), sw.transform(Xw_te)
print('model            raw     scaled')
for name, mk in [('GaussianNB', lambda: GaussianNB()), ('KNN (K=3)', lambda: KNeighborsClassifier(3))]:
    a = mk().fit(Xw_tr, yw_tr).score(Xw_te, yw_te)
    b = mk().fit(Xw_tr_s, yw_tr).score(Xw_te_s, yw_te)
    print(f'{name:16} {a:.4f}  {b:.4f}')

model            raw     scaled
GaussianNB       0.9815  0.9815
KNN (K=3)        0.6667  0.9444


### Student Practice 4
1. Explain in one line why Gaussian NB does not care about feature units. (Look at $(x-\mu)^2/\sigma^2$: what happens to it if you change cm to mm?)
2. One split can be lucky. Report GaussianNB accuracy on the full Wine data with 5-fold cross-validation (mean and std). Why is it safe to skip the scaler here?

In [13]:
# Student Practice 4: Solution

# 1. Explanation of unit invariance
print("1. Explanation of unit invariance:")
print("   In the Gaussian likelihood exponent, (x - mu)^2 / sigma^2, converting units by scale k (e.g. cm to mm)")
print("   scales differences (kx - k*mu)^2 = k^2*(x - mu)^2 and variance k^2*sigma^2, which cancel out completely.")
print("   The prefactor 1 / (sqrt(2*pi)*k*sigma) adds an identical offset -ln(k) across all classes,")
print("   which cancels out when comparing class posteriors. Hence, GaussianNB is strictly scale-invariant.")

# 2. 5-fold cross-validation on full Wine dataset
cv_scores = cross_val_score(GaussianNB(), Xw, yw, cv=5)
print("\n2. GaussianNB 5-fold CV on Wine:")
print(f"   Fold accuracies: {cv_scores.round(4)}")
print(f"   Mean accuracy  : {cv_scores.mean():.4f}")
print(f"   Std deviation  : {cv_scores.std():.4f}")
print("   Why safe to skip scaler: Because GaussianNB estimates per-feature, per-class mean and variance,")
print("   scaling features beforehand does not change likelihood ratios or posterior class decisions.")


1. Explanation of unit invariance:
   In the Gaussian likelihood exponent, (x - mu)^2 / sigma^2, converting units by scale k (e.g. cm to mm)
   scales differences (kx - k*mu)^2 = k^2*(x - mu)^2 and variance k^2*sigma^2, which cancel out completely.
   The prefactor 1 / (sqrt(2*pi)*k*sigma) adds an identical offset -ln(k) across all classes,
   which cancels out when comparing class posteriors. Hence, GaussianNB is strictly scale-invariant.

2. GaussianNB 5-fold CV on Wine:
   Fold accuracies: [0.9444 0.9722 0.9722 0.9429 1.    ]
   Mean accuracy  : 0.9663
   Std deviation  : 0.0211
   Why safe to skip scaler: Because GaussianNB estimates per-feature, per-class mean and variance,
   scaling features beforehand does not change likelihood ratios or posterior class decisions.


# Mini Project A: Naive Bayes on Digits (Image Track — continuing from Weeks 7, 8 and 9)
Same split as before (`test_size=0.3, random_state=1, stratify`). Baselines you already have: **KNN (scaled, K=3) = 0.9796** and **Tree (entropy, full depth, unscaled) = 0.8815**. Question: how does Naive Bayes compare, and does Week 9's PCA help it?

In [14]:
digits = load_digits()
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    digits.data, digits.target, test_size=0.3, random_state=1, stratify=digits.target)
print('pixels with zero variance in train:', int((Xd_train.std(0) == 0).sum()), 'of 64')
print('GaussianNB test accuracy:', round(GaussianNB().fit(Xd_train, yd_train).score(Xd_test, yd_test), 4))

pixels with zero variance in train: 3 of 64
GaussianNB test accuracy: 0.8315


### Predict-then-run
Before running the cell above, guess: will GaussianNB score **above both** baselines, **between** them, or **below both**? The 64 pixels are clearly *not* independent given the digit. Is that a problem for NB?

### Student Practice 5
1. Fit `MultinomialNB()` (treat pixel intensity as a count) and `BernoulliNB(binarize=8)` (ink / no ink) on the raw pixels. Report test accuracy for both.
2. GaussianNB has one knob, `var_smoothing`. For values `[1e-9, 1e-3, 1e-2, 1e-1, 1]`, use 5-fold cross-validation **on the training set only**, pick the best, then report its test accuracy **once**. (Why not pick the value with the best test accuracy?)
3. Using that best model, which two digits are confused most often?

In [15]:
# Student Practice 5: Solution

# 1. Fit MultinomialNB and BernoulliNB on raw pixels
mnb = MultinomialNB().fit(Xd_train, yd_train)
bnb = BernoulliNB(binarize=8).fit(Xd_train, yd_train)
acc_mnb = mnb.score(Xd_test, yd_test)
acc_bnb = bnb.score(Xd_test, yd_test)

print("1. Raw Pixel Test Accuracies:")
print(f"   MultinomialNB (pixel intensity as counts) : {acc_mnb:.4f} ({acc_mnb*100:.2f}%)")
print(f"   BernoulliNB (binarized at threshold=8)    : {acc_bnb:.4f} ({acc_bnb*100:.2f}%)")

# 2. Tuning var_smoothing on training set only using 5-fold CV
var_values = [1e-9, 1e-3, 1e-2, 1e-1, 1]
cv_means = {}

print("\n2. Tuning var_smoothing on train set via 5-fold CV:")
for v in var_values:
    score = cross_val_score(GaussianNB(var_smoothing=v), Xd_train, yd_train, cv=5).mean()
    cv_means[v] = score
    print(f"   var_smoothing = {v:<6}: CV score = {score:.4f}")

best_var = max(cv_means, key=cv_means.get)
best_gnb = GaussianNB(var_smoothing=best_var).fit(Xd_train, yd_train)
test_acc_best = best_gnb.score(Xd_test, yd_test)

print(f"\n   Best var_smoothing: {best_var} (CV accuracy = {cv_means[best_var]:.4f})")
print(f"   Test accuracy of best model: {test_acc_best:.4f} ({test_acc_best*100:.2f}%)")
print("   Why not pick using test accuracy? Selecting hyperparameters on the test set causes data leakage /")
print("   test snooping, yielding an optimistically biased evaluation that will not reflect generalisation performance.")

# 3. Most confused digit pair
y_pred = best_gnb.predict(Xd_test)
cm_digits = confusion_matrix(yd_test, y_pred)
np.fill_diagonal(cm_digits, 0)
true_digit, pred_digit = np.unravel_index(np.argmax(cm_digits), cm_digits.shape)

print("\n3. Confusion Analysis:")
print(f"   Most confused pair: true digit {true_digit} misclassified as {pred_digit} ({cm_digits[true_digit, pred_digit]} times).")


1. Raw Pixel Test Accuracies:
   MultinomialNB (pixel intensity as counts) : 0.8963 (89.63%)
   BernoulliNB (binarized at threshold=8)    : 0.8963 (89.63%)

2. Tuning var_smoothing on train set via 5-fold CV:
   var_smoothing = 1e-09 : CV score = 0.8187
   var_smoothing = 0.001 : CV score = 0.9030
   var_smoothing = 0.01  : CV score = 0.9173
   var_smoothing = 0.1   : CV score = 0.9141
   var_smoothing = 1     : CV score = 0.9046

   Best var_smoothing: 0.01 (CV accuracy = 0.9173)
   Test accuracy of best model: 0.9296 (92.96%)
   Why not pick using test accuracy? Selecting hyperparameters on the test set causes data leakage /
   test snooping, yielding an optimistically biased evaluation that will not reflect generalisation performance.

3. Confusion Analysis:
   Most confused pair: true digit 1 misclassified as 8 (6 times).


### Deliberate error: negative inputs
Since Week 9 we standardise pixels before PCA. Run the cell: what breaks, and why?

In [16]:
sc = StandardScaler().fit(Xd_train)                    # fit on TRAIN only
Xtr_s, Xte_s = sc.transform(Xd_train), sc.transform(Xd_test)
try:
    MultinomialNB().fit(Xtr_s, yd_train)
except ValueError as e:
    print('ValueError:', e)
print('GaussianNB on scaled pixels:', round(GaussianNB().fit(Xtr_s, yd_train).score(Xte_s, yd_test), 4), ' (raw pixels gave 0.8315)')

ValueError: Negative values in data passed to MultinomialNB (input X).
GaussianNB on scaled pixels: 0.7759  (raw pixels gave 0.8315)


### Link to Week 9: does decorrelating the pixels help?
Naive Bayes assumes features are independent given the class. PCA scores are *uncorrelated by construction*. Question: does PCA before NB help?

### Student Practice 6
1. Compute the mean absolute off-diagonal correlation (a) of the non-constant raw pixels and (b) of the first 30 PCA scores of `Xtr_s`.
2. For k in `[10, 20, 30, 40, 64]` fit `PCA(k)` on `Xtr_s` (train only), then `GaussianNB` on the scores, and report test accuracy.
3. `PCA(64)` is only a rotation of the scaled pixels: no information is lost. So why does its accuracy differ from GaussianNB on scaled pixels (above)?

In [17]:
# Student Practice 6: Solution

# 1. Mean absolute off-diagonal correlation
# (a) Non-constant raw pixels in training set
non_const_mask = Xd_train.std(axis=0) > 0
raw_non_const = Xd_train[:, non_const_mask]
corr_raw = np.corrcoef(raw_non_const, rowvar=False)
mask_raw = ~np.eye(corr_raw.shape[0], dtype=bool)
mean_abs_corr_raw = np.mean(np.abs(corr_raw[mask_raw]))

# (b) First 30 PCA scores of Xtr_s
pca30 = PCA(n_components=30).fit(Xtr_s)
Ztr_30 = pca30.transform(Xtr_s)
corr_pca = np.corrcoef(Ztr_30, rowvar=False)
mask_pca = ~np.eye(corr_pca.shape[0], dtype=bool)
mean_abs_corr_pca = np.mean(np.abs(corr_pca[mask_pca]))

print("1. Mean absolute off-diagonal correlation:")
print(f"   (a) Raw non-constant pixels : {mean_abs_corr_raw:.4f}")
print(f"   (b) First 30 PCA scores     : {mean_abs_corr_pca:.4e} (effectively 0)")

# 2. GaussianNB on PCA(k) scores
print("\n2. Test accuracy for k components with PCA + GaussianNB:")
for k in [10, 20, 30, 40, 64]:
    pca_k = PCA(n_components=k).fit(Xtr_s)
    Ztr_k = pca_k.transform(Xtr_s)
    Zte_k = pca_k.transform(Xte_s)
    acc_k = GaussianNB().fit(Ztr_k, yd_train).score(Zte_k, yd_test)
    print(f"   k = {k:2d}: Test Accuracy = {acc_k:.4f} ({acc_k*100:.2f}%)")

# 3. Why PCA(64) differs from GaussianNB on scaled pixels
acc_scaled = GaussianNB().fit(Xtr_s, yd_train).score(Xte_s, yd_test)
print(f"\n3. Accuracy of GaussianNB on scaled pixels directly: {acc_scaled:.4f}")
print("   Why PCA(64) gives higher accuracy (0.9185 vs 0.7759) despite being an exact rotation:")
print("   GaussianNB assumes features are conditionally independent, fitting an axis-aligned (diagonal covariance)")
print("   Gaussian. In raw/scaled pixel space, neighbouring pixels are strongly correlated, violating this assumption.")
print("   PCA rotates the data into principal axes that are globally decorrelated, allowing the axis-aligned Gaussian")
print("   model to fit the underlying distribution far better. GaussianNB is not rotation-invariant!")


1. Mean absolute off-diagonal correlation:
   (a) Raw non-constant pixels : 0.1257
   (b) First 30 PCA scores     : 3.7684e-16 (effectively 0)

2. Test accuracy for k components with PCA + GaussianNB:
   k = 10: Test Accuracy = 0.8296 (82.96%)
   k = 20: Test Accuracy = 0.9148 (91.48%)
   k = 30: Test Accuracy = 0.9259 (92.59%)
   k = 40: Test Accuracy = 0.9222 (92.22%)
   k = 64: Test Accuracy = 0.9185 (91.85%)

3. Accuracy of GaussianNB on scaled pixels directly: 0.7759
   Why PCA(64) gives higher accuracy (0.9185 vs 0.7759) despite being an exact rotation:
   GaussianNB assumes features are conditionally independent, fitting an axis-aligned (diagonal covariance)
   Gaussian. In raw/scaled pixel space, neighbouring pixels are strongly correlated, violating this assumption.
   PCA rotates the data into principal axes that are globally decorrelated, allowing the axis-aligned Gaussian
   model to fit the underlying distribution far better. GaussianNB is not rotation-invariant!


# Mini Project B: Naive Bayes on the Six Sentences (NLP Track — continuing from Weeks 7, 8 and 9)
Same sentences and bag-of-words vectors. Naive Bayes is the classic text classifier (spam filters!). In Week 9, PCA failed to find the topics because *the* and *and* dominated the variance. Question: does NB do better?

In [18]:
sentences = [
    'the cricket team won the match',
    'the batsman scored a century today',
    'chop the onions and boil the pasta',
    'add salt and simmer the curry',
    'the new phone has a faster processor',
    'the laptop battery lasts all day',
]
labels = ['sports', 'sports', 'cooking', 'cooking', 'tech', 'tech']
vocab = sorted(set(w for s in sentences for w in s.lower().split()))
def vec(s): return [s.lower().split().count(w) for w in vocab]       # words outside the vocabulary are ignored
X_text = np.array([vec(s) for s in sentences])
print('matrix shape (docs x vocab):', X_text.shape)
nb_text = MultinomialNB(alpha=1).fit(X_text, labels)
print('classes:', nb_text.classes_, ' priors:', np.exp(nb_text.class_log_prior_))

matrix shape (docs x vocab): (6, 29)
classes: ['cooking' 'sports' 'tech']  priors: [0.3333 0.3333 0.3333]


### Predict-then-run
Multinomial NB with smoothing uses $P(w\mid C)=\dfrac{\text{count}(w,C)+\alpha}{\text{total words in }C+\alpha\,|V|}$ with $|V|=29$.
(a) How many word tokens are in the *sports* class (two sentences)? (b) Which class will NB give to **'the new curry'**? Is it confident?

### Student Practice 7
1. Predict the five sentences below with `nb_text` (`alpha=1`) and print the class probabilities. Accuracy?
   `'the team scored a century'` (sports), `'boil the pasta with salt'` (cooking), `'the phone battery lasts a day'` (tech), `'the match today'` (sports), `'the new curry'` (cooking)
2. By hand: for `'the team scored a century'` compute the three class scores from the count formula above and normalise. Compare with `predict_proba`.
3. Print the 5 words with the highest $P(w\mid C)$ in each class. Is *the* near the top everywhere? Why does that **not** hurt Naive Bayes, when it did hurt PCA?
4. Why is the probability for **'the new curry'** so spread out?

In [19]:
# Student Practice 7: Solution

# 1. Predict the five sentences with nb_text (alpha=1)
test_sentences = [
    ('the team scored a century', 'sports'),
    ('boil the pasta with salt', 'cooking'),
    ('the phone battery lasts a day', 'tech'),
    ('the match today', 'sports'),
    ('the new curry', 'cooking')
]

print("1. Predictions and Class Probabilities:")
preds = []
for s, true_lbl in test_sentences:
    v = [vec(s)]
    pred = nb_text.predict(v)[0]
    probs = nb_text.predict_proba(v)[0]
    preds.append(pred)
    prob_str = ", ".join([f"{c}: {p:.4f}" for c, p in zip(nb_text.classes_, probs)])
    print(f"   '{s}'\n      -> Pred: {pred:<7} | True: {true_lbl:<7} | Probs: {{{prob_str}}}")

acc_text = np.mean([p == t[1] for p, t in zip(preds, test_sentences)])
print(f"   Accuracy: {acc_text:.4f} ({acc_text*100:.1f}%)")

# 2. By hand calculation for 'the team scored a century'
words = 'the team scored a century'.split()
print("\n2. By hand calculation for 'the team scored a century':")
print(f"   Vocabulary size |V| = {len(vocab)}")
hand_scores = {}
for c in nb_text.classes_:
    c_mask = np.array(labels) == c
    tot_words_c = X_text[c_mask].sum()
    score = 1.0 / 3.0  # prior
    fracs = []
    for w in words:
        w_idx = vocab.index(w)
        w_cnt = X_text[c_mask, w_idx].sum()
        p_w = (w_cnt + 1) / (tot_words_c + len(vocab))
        fracs.append(f"({w_cnt}+1)/({tot_words_c}+29)")
        score *= p_w
    hand_scores[c] = score
    print(f"   Class {c:7}: {' * '.join(fracs)} * (1/3) = {score:.6e}")

total_hand = sum(hand_scores.values())
norm_hand = {c: hand_scores[c] / total_hand for c in hand_scores}
print(f"   Normalised manual posterior: { {c: round(v, 4) for c, v in norm_hand.items()} }")
sklearn_proba = dict(zip(nb_text.classes_, nb_text.predict_proba([vec('the team scored a century')])[0].round(4)))
print(f"   sklearn predict_proba      : {sklearn_proba}")
print("   Manual calculation matches sklearn predict_proba exactly!")

# 3. Top 5 words per class with highest P(w|C)
print("\n3. Top 5 words per class by P(w|C):")
feat_probs = np.exp(nb_text.feature_log_prob_)
for i, c in enumerate(nb_text.classes_):
    top5_idx = np.argsort(feat_probs[i])[::-1][:5]
    top5_words = [(vocab[idx], round(feat_probs[i, idx], 4)) for idx in top5_idx]
    print(f"   {c:7}: {top5_words}")
print("   Is 'the' near the top everywhere? Yes ('the' is #1 in all classes).")
print("   Why this does NOT hurt Naive Bayes (unlike PCA):")
print("   In PCA, 'the' dominated total variance across documents, misdirecting principal axes.")
print("   In Naive Bayes, classification depends on likelihood RATIOS between classes. Because 'the' appears")
print("   with virtually identical probability in all classes, it cancels out across classes and does not bias decisions.")

# 4. Why 'the new curry' probability is spread out
print("\n4. Why is the probability for 'the new curry' so spread out?")
print("   'the' is neutral/common to all classes, 'new' appears only in tech, and 'curry' appears only in cooking.")
print("   With Laplace smoothing, the model sees conflicting signals: 1 evidence token for cooking and 1 for tech,")
print("   while sports gets smoothed pseudo-counts for both. Hence cooking receives 43.7%, tech 32.8%, and sports 23.5%.")


1. Predictions and Class Probabilities:
   'the team scored a century'
      -> Pred: sports  | True: sports  | Probs: {cooking: 0.0487, sports: 0.8783, tech: 0.0730}
   'boil the pasta with salt'
      -> Pred: cooking | True: cooking | Probs: {cooking: 0.8121, sports: 0.1118, tech: 0.0761}
   'the phone battery lasts a day'
      -> Pred: tech    | True: tech    | Probs: {cooking: 0.0366, sports: 0.0846, tech: 0.8788}
   'the match today'
      -> Pred: sports  | True: sports  | Probs: {cooking: 0.1653, sports: 0.7107, tech: 0.1240}
   'the new curry'
      -> Pred: cooking | True: cooking | Probs: {cooking: 0.4372, sports: 0.2350, tech: 0.3279}
   Accuracy: 1.0000 (100.0%)

2. By hand calculation for 'the team scored a century':
   Vocabulary size |V| = 29
   Class cooking: (3+1)/(13+29) * (0+1)/(13+29) * (0+1)/(13+29) * (0+1)/(13+29) * (0+1)/(13+29) * (1/3) = 1.020216e-08
   Class sports : (3+1)/(12+29) * (1+1)/(12+29) * (1+1)/(12+29) * (1+1)/(12+29) * (1+1)/(12+29) * (1/3) = 1.841

### Deliberate error: almost no smoothing
`'the cricket phone'` mixes a sports word with a tech word. Run the cell and compare how *confident* each model is.

In [20]:
for a in [1e-10, 1.0]:
    m = MultinomialNB(alpha=a).fit(X_text, labels)
    print(f'alpha={a:<6}', dict(zip(m.classes_.tolist(), m.predict_proba([vec('the cricket phone')])[0].round(3).tolist())))

alpha=1e-10  {'cooking': 0.0, 'sports': 0.656, 'tech': 0.344}
alpha=1.0    {'cooking': 0.215, 'sports': 0.462, 'tech': 0.323}


## Part 6: Reflection (write 2–3 lines each)

1. **In your own words: what is the "naive" assumption, and what does it make easy?**  
   The "naive" assumption states that all features are conditionally independent given the class label ($P(x_1, \dots, x_d \mid C) = \prod_{j=1}^d P(x_j \mid C)$). It dramatically simplifies learning because instead of estimating a full exponential joint distribution ($O(k^d)$ parameters), the model only needs to estimate $d$ separate 1D distributions ($O(kd)$ parameters), requiring far less data and compute.

2. **What is the zero-frequency problem? Show it with the Play Tennis example and say how Laplace smoothing fixes it.**  
   If a feature value never co-occurs with a class in the training data (e.g. Outlook='Overcast' with Play='No'), its estimated likelihood $P(\text{Overcast}\mid\text{No})$ is 0. Because features are multiplied, this single zero forces the entire class posterior to 0 regardless of all other evidence. Laplace smoothing fixes this by adding a pseudo-count $\alpha > 0$ to numerators and $\alpha \cdot k$ to denominators, ensuring all probabilities remain strictly positive.

3. **Why do real implementations add log-probabilities instead of multiplying probabilities?**  
   Multiplying many probabilities ($0 < p < 1$) over dozens or hundreds of features causes severe floating-point underflow, collapsing numbers to 0.0 in finite precision. Taking logarithms converts multiplication into addition ($\sum \log P$), which is numerically stable, computationally efficient, and preserves monotonic class rankings.

4. **Wine: scaling changed KNN but not Gaussian NB. What does that tell you about how each model uses the features?**  
   KNN computes geometric Euclidean distances across all features together, so large-scale features (like Proline) dominate the distance metric unless standardized. In contrast, Gaussian NB evaluates each feature independently through its class z-score $(x - \mu)^2 / \sigma^2$; multiplying a feature by a constant scales difference and standard deviation identically, making Gaussian NB intrinsically scale-invariant.

5. **Digits: PCA helped NB but hurt the tree (Week 9). Explain using the independence assumption.**  
   Neighboring image pixels are strongly correlated, violating Naive Bayes' independence assumption; PCA rotates the data onto orthogonal axes that are globally decorrelated, allowing NB's diagonal Gaussian model to fit much better. Conversely, Decision Trees split cleanly along original individual pixel boundaries (ink vs. no ink); PCA projects all pixels into dense linear combinations, which scatters local threshold patterns and hurts tree splits.

6. **Naive Bayes is fast and needs little data, but its probabilities are often overconfident. When would you still choose it?**  
   Naive Bayes is an excellent choice for high-dimensional, sparse text and document classification (e.g. spam filtering, sentiment analysis, topic tagging), streaming environments where ultra-fast $O(d)$ inference and incremental updates are required, and low-data regimes where complex models are prone to overfitting.

7. **Capstone link (Milestone, Week 10): will Naive Bayes belong in your pipeline, at least as a baseline? If yes, add its accuracy to your comparison table in your repo's `notebooks/` folder; if no, justify.**  
   Yes, Naive Bayes (MultinomialNB for text/count data, GaussianNB for continuous data) serves as an essential, high-speed baseline in our machine learning pipeline. It provides a zero-tuning, highly interpretable lower-bound benchmark to verify whether more complex non-linear architectures (e.g. Random Forests, Gradient Boosted Trees, Neural Networks) justify their added computational overhead.

*Submit this notebook to GitHub under `labs/week10_naive_bayes/`.*